In [ ]:
import sys

sys.path.append("..")
sys.path.insert(0, "../../wildlife-datasets")
sys.path.insert(0, "../../wildlife-tools")

import pickle
import warnings
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from wildlife_tools.inference import TopkClassifier

from animal_re_id.datasets import SeaTurtleID2022, combine_datasets_twe
from utils import Data, get_info, merge_data, plot_best_scores

warnings.filterwarnings("ignore")

In [ ]:
file_name = "../notebooks/individuals.csv"
root = "/data/wildlife_datasets/turtle_watch/citizen"
root_heads = "/data/wildlife_datasets/turtle_watch/PROFILES"
root_flippers_f = "/data/wildlife_datasets/turtle_watch/TWE_flipperprofiles_registered ind_LR_front"
root_flippers_r = "/data/wildlife_datasets/turtle_watch/TWE_flipperprofiles_registered ind_LR_rear"
root_new = "/data/wildlife_datasets/turtle_watch"
roots_new = [
    "2017 PICS + DATASHEET",
    "2018 PICS + DATASHEET",
    "2019 PICS + DATASHEET",
    "2020 PICS + DATASHEET",
    "2021 PICS + DATASHEET",
    "2022 PICS + online database",
    "2023 PICS + online database",
    "2024 PICS + online database",
    "2025 PICS + online database",
]
root_results = "/data/wildlife_datasets/similarity"

In [ ]:
dataset_name = "twe"
# dataset_name = "seaturtleid2022"

matching_parts = ["heads", "front flippers", "rear flippers"]
img_loads = ["bbox"]

ks = [1, 10, 100]
Bs = [5, 10, 50, 100]
matching_merged = "merged"

In [ ]:
if dataset_name == "twe":
    dataset_true = combine_datasets_twe(root_new, roots_new, file_name, load_segmentation=False)
    mask1 = dataset_true.metadata["identity"] != "unknown"
    mask2 = ~dataset_true.metadata["identity"].isnull()
    dataset_true = dataset_true.get_subset(mask1 & mask2)
    dataset_true.metadata["year"] = dataset_true.metadata["path"].str[:4].astype(int)
    split_years = np.arange(2018, 2026).tolist()
    pipeline_funs = [
        ("miew", 1),
        ("miew-aliked", 1.25),
    ]
    y_maxs = [8, 0.04]
elif dataset_name == "seaturtleid2022":
    root = "/data/wildlife_datasets/data/SeaTurtleID2022"
    dataset_true = SeaTurtleID2022(root, False).dataset
    split_years = np.arange(2011, 2023).tolist()
    pipeline_funs = [
        ("dino", 1),
        ("dino-aliked", 1.25),
    ]
    y_maxs = [40, 0.07]
else:
    raise ValueError("Dataset now known")

identities, n_encounters_all, n_encounters_known = get_info(dataset_true, split_years)

In [ ]:
k_max = max(ks)
pars_all = list(
    product(
        img_loads,
        Bs,
        pipeline_funs,
        [False, True],
        split_years,
    )
)

In [ ]:
datas = {pars: {} for pars in pars_all}
for pars in pars_all:
    img_load, B, (name, head_multiplier), match_opposite_sides, split_year = pars

    for matching_part in matching_parts:
        # Load the saved variables
        file_name_save = f"{dataset_name}_{name}_{B}_{img_load}_{matching_part}_{match_opposite_sides}_{split_year}"
        with open(f"{root_results}/{dataset_name}/{file_name_save}.pkl", "rb") as file:
            saved_vars = pickle.load(file)
        similarity = saved_vars["similarity"]
        query = saved_vars["query"]
        database = saved_vars["database"]
        
        # Merge the rows of the similarity matrix by encounters
        groups = query.metadata.groupby("encounter_id").indices
        unique_encounters = list(groups.keys())
        similarity_encounters = np.vstack([similarity[idx].max(axis=0) for idx in groups.values()])

        # Get the predictions including scores for encounters
        labels_database = database.metadata["identity"].to_numpy()
        classifier = TopkClassifier(labels_database, k=k_max, return_all=True)
        y_preds, scores, _ = classifier(similarity_encounters)

        # Multiple the scores for heads
        if matching_part == "heads":
            scores *= head_multiplier

        # Save them as a dictionary
        scores_part = {x: y for x, y in zip(unique_encounters, scores)}
        preds_part = {x: y for x, y in zip(unique_encounters, y_preds)}
        datas[pars][matching_part] = Data(scores_part, preds_part, matching_part, k_max=k_max)

In [ ]:
parts_all = {}

for pars in pars_all:
    data_selection = {m: datas[pars][m] for m in matching_parts}
    data_merged = merge_data(data_selection, k_max)
    datas[pars][matching_merged] = data_merged

In [ ]:
accuracy_all = {}
for pars in pars_all:
    img_load, B, (name, _), match_opposite_sides, split_year = pars
    for matching_part in matching_parts + [matching_merged]:
        data = datas[pars][matching_part]
        for k in ks:
            accuracy = data.compute_matches(k, identities) / n_encounters_known[split_year]
            accuracy_all[(img_load, B, matching_part, name, match_opposite_sides, split_year, k)] = accuracy

In [ ]:
accuracy_all = []
for pars in pars_all:
    img_load, B, (name, _), match_opposite_sides, split_year = pars
    for matching_part in matching_parts + [matching_merged]:
        data = datas[pars][matching_part]
        for k in ks:
            accuracy = data.compute_matches(k, identities) / n_encounters_known[split_year]
            accuracy_all.append({
                "img_load": img_load,
                "B": B,
                "matching_part": matching_part, 
                "name": name,
                "match_opposite_sides": match_opposite_sides, 
                "split_year": split_year,
                "k": k,
                "accuracy": accuracy
            })

In [ ]:
df = pd.DataFrame(accuracy_all)
tables = {}
for matching_part in df["matching_part"].unique():
    for k in ks:
        sub = df[(df["matching_part"] == matching_part) & (df["k"] == k)]
        table = sub.pivot_table(
            index=["B", "img_load", "name", "match_opposite_sides"],
            columns="split_year",
            values="accuracy",
        ).sort_index()
        table["average"] = table.mean(axis=1)
        tables[(matching_part, k)] = table

In [ ]:
k = 1
table_average = pd.DataFrame()
for matching_part in matching_parts + [matching_merged]:
    t = tables[(matching_part, k)].copy()
    t = t.drop(split_years, axis=1).rename({"average": matching_part}, axis=1)
    table_average = pd.concat((table_average, t), axis=1)

table_average

In [ ]:
name_desired, _ = pipeline_funs[-1]
best_scores = []
for pars in pars_all:
    img_load, B, (name, _), match_opposite_sides, split_year = pars
    if img_load != "bbox" or B != max(Bs) or name != name_desired or match_opposite_sides:
        continue

    data = datas[pars]["merged"]
    best_score = data.compute_matches_by_part(1, identities)
    best_scores.extend(best_score)

plot_best_scores(best_scores, matching_parts, y_max=y_maxs[-1])
plt.savefig(f"{dataset_name}_merged.png", dpi=300, bbox_inches="tight")

In [ ]:

for (name_desired, _), y_max in zip(pipeline_funs, y_maxs):
    best_scores = []
    for pars in pars_all:
        img_load, B, (name, _), match_opposite_sides, split_year = pars
        if img_load != "bbox" or B != max(Bs) or name != name_desired or match_opposite_sides:
            continue

        for matching_part in matching_parts:
            data = datas[pars][matching_part]
            best_score = data.compute_matches_by_part(1, identities)
            best_scores.extend(best_score)

    plot_best_scores(best_scores, matching_parts, y_max)
    plt.savefig(f"{dataset_name}_{name_desired}.png", dpi=300, bbox_inches="tight")